# Paper Figures Generation

This notebook generates high-quality figures for the empirical transfer learning paper:
- **Figure 1**: Within-Domain Baseline Performance
- **Figure 2**: Zero-Shot Transfer Performance Heatmap
- **Figure 3**: Transfer Learning Improvement Analysis

Based on experimental results from NASA, PHM2010, and NATURE2024 datasets.

## 1. Import Libraries and Configuration

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

# Set plotting style for publication-quality figures
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("colorblind")

# Configure matplotlib for better font rendering
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 12

# Define color scheme consistent with paper
COLOR_1D_CONV = '#2E5A87'  # Dark blue for 1D Conv
COLOR_LSTM = '#52A552'     # Green for LSTM
COLOR_BASELINE = '#808080' # Gray for baselines

print("✓ Libraries imported successfully")

## 2. Define Paths and Load Experiment Data

In [ ]:
# Define the parent directory for experiments (UPDATE THIS PATH)
experiments_base_dir = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/experiments'
summary_csv_path = os.path.join(experiments_base_dir, 'experiments_evaluation_summary.csv')

# Output directory for generated figures
figures_output_dir = os.path.join(experiments_base_dir, 'paper_figures')
os.makedirs(figures_output_dir, exist_ok=True)

print(f"Experiments base directory: {experiments_base_dir}")
print(f"Summary CSV path: {summary_csv_path}")
print(f"Figures will be saved to: {figures_output_dir}")

In [ ]:
# Load experiment data from summary CSV
if os.path.exists(summary_csv_path):
    experiments_df = pd.read_csv(summary_csv_path)
    print(f"Loaded {len(experiments_df)} rows from summary CSV")
else:
    raise FileNotFoundError(f"Summary CSV not found at {summary_csv_path}")

# Filter out autoencoders
experiments_df = experiments_df[
    (experiments_df['ModelType'] != 'autoencoder') & 
    (~experiments_df['Experiment Name'].str.contains('autoencoder', case=False))
].copy()

# Sort by experiment name
# Note: Assuming 'Train Cases' and 'Test Cases' columns exist in the CSV. 
# If not, we can sort by 'Experiment Name' only.
sort_cols = ['Experiment Name']
if 'Train Cases' in experiments_df.columns and 'Test Cases' in experiments_df.columns:
    sort_cols.extend(['Train Cases', 'Test Cases'])

experiments_df = experiments_df.sort_values(by=sort_cols).reset_index(drop=True)

print(f"\nFiltered to {len(experiments_df)} experiment configurations (excluding autoencoders)")
print(f"\nDatasets: {experiments_df['BaseData'].unique()}")
print(f"Model Types: {experiments_df['ModelType'].unique()}")
print(f"Learning Modes: {experiments_df['LearningMode'].unique()}")

experiments_df.head()

## 3. Load Evaluation Results

In [ ]:
# Display summary of loaded results (already present in CSV)
print(f"\nEvaluation results available for {experiments_df['Accuracy'].notna().sum()} / {len(experiments_df)} experiments")
print(f"\nAccuracy statistics:")
print(experiments_df.groupby(['BaseData', 'ModelType', 'LearningMode'])['Accuracy'].agg(['count', 'mean', 'std']))

## 4. Data Preparation and Mapping

In [ ]:
# Create clean dataset names for visualization
dataset_name_map = {
    'nasa_milling': 'NASA',
    'phm_milling': 'PHM2010',
    'nature_milling': 'NATURE2024',
    # Add short names found in CSV TransferData
    'nasa': 'NASA',
    'phm': 'PHM2010',
    'nature': 'NATURE2024'
}

model_name_map = {
    '1dconv': '1D Conv',
    'lstm': 'LSTM'
}

mode_name_map = {
    'baseTraining': 'Base',
    'inference': 'Inference',
    'transferLearning': 'Transfer Learning'
}

# Apply mappings
experiments_df['Dataset_Clean'] = experiments_df['BaseData'].map(dataset_name_map)
experiments_df['Model_Clean'] = experiments_df['ModelType'].map(model_name_map)
experiments_df['Mode_Clean'] = experiments_df['LearningMode'].map(mode_name_map)
experiments_df['Transfer_Clean'] = experiments_df['TransferData'].map(dataset_name_map)

print("✓ Data mappings applied")
experiments_df[['Dataset_Clean', 'Model_Clean', 'Mode_Clean', 'Transfer_Clean', 'Accuracy']].head(10)

## 5. Figure 1: Within-Domain Baseline Performance

This figure shows the baseline performance for each dataset when trained and tested on the same domain using 5-fold cross-validation.

In [ ]:
# Filter for base training experiments only
base_training_df = experiments_df[
    (experiments_df['LearningMode'] == 'baseTraining') &
    (experiments_df['Accuracy'].notna())
].copy()

# Calculate mean and std for each dataset-model combination
base_stats = base_training_df.groupby(['Dataset_Clean', 'Model_Clean'])['Accuracy'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('count', 'count')
]).reset_index()

print("Base Training Performance Statistics:")
print(base_stats)

# Verify we have the expected datasets
expected_datasets = ['NASA', 'PHM2010', 'NATURE2024']
available_datasets = base_stats['Dataset_Clean'].unique()
print(f"\nDatasets available: {available_datasets}")
print(f"Expected datasets: {expected_datasets}")

In [ ]:
# Create Figure 1: Base Training Performance
fig, ax = plt.subplots(figsize=(8, 5))

# Define x positions for datasets
datasets_ordered = ['NASA', 'NATURE2024', 'PHM2010']
x_pos = np.arange(len(datasets_ordered))
width = 0.35  # Width of bars

# Prepare data for plotting
conv_means = []
conv_stds = []
lstm_means = []
lstm_stds = []

for dataset in datasets_ordered:
    # 1D Conv
    conv_data = base_stats[(base_stats['Dataset_Clean'] == dataset) & 
                           (base_stats['Model_Clean'] == '1D Conv')]
    if not conv_data.empty:
        conv_means.append(conv_data['mean'].values[0])
        conv_stds.append(conv_data['std'].values[0])
    else:
        conv_means.append(0)
        conv_stds.append(0)
    
    # LSTM
    lstm_data = base_stats[(base_stats['Dataset_Clean'] == dataset) & 
                           (base_stats['Model_Clean'] == 'LSTM')]
    if not lstm_data.empty:
        lstm_means.append(lstm_data['mean'].values[0])
        lstm_stds.append(lstm_data['std'].values[0])
    else:
        lstm_means.append(0)
        lstm_stds.append(0)

# Plot bars
bars1 = ax.bar(x_pos - width/2, conv_means, width, 
               yerr=conv_stds, 
               label='1D Conv', 
               color=COLOR_1D_CONV, 
               alpha=0.8,
               capsize=5,
               error_kw={'linewidth': 1.5, 'elinewidth': 1.5})

bars2 = ax.bar(x_pos + width/2, lstm_means, width, 
               yerr=lstm_stds, 
               label='LSTM', 
               color=COLOR_LSTM, 
               alpha=0.8,
               capsize=5,
               error_kw={'linewidth': 1.5, 'elinewidth': 1.5})

# Add horizontal line at 0.5 (random baseline)
ax.axhline(y=0.5, color=COLOR_BASELINE, linestyle='--', linewidth=1, 
           label='Random Baseline', alpha=0.6)

# Customize plot
ax.set_xlabel('Dataset', fontweight='bold')
ax.set_ylabel('Classification Accuracy', fontweight='bold')
ax.set_title('Within-Domain Baseline Performance (5-fold CV)', fontweight='bold', pad=15)
ax.set_xticks(x_pos)
ax.set_xticklabels(datasets_ordered)
ax.set_ylim([0, 1.0])
ax.legend(loc='upper left', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle=':')

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        if height > 0:  # Only add label if there's data
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', fontsize=8)

add_value_labels(bars1)
add_value_labels(bars2)

plt.tight_layout()

# Save figure
fig1_path = os.path.join(figures_output_dir, 'figure1_base_training_performance.pdf')
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
fig1_png_path = os.path.join(figures_output_dir, 'figure1_base_training_performance.png')
plt.savefig(fig1_png_path, dpi=300, bbox_inches='tight')

print(f"✓ Figure 1 saved to: {fig1_path}")
plt.show()

## 6. Figure 2: Zero-Shot Transfer Performance Heatmap

This figure visualizes the performance of models trained on one dataset and tested directly on another (zero-shot inference), showing the complete transfer matrix.

In [ ]:
# Filter for inference (zero-shot) experiments
inference_df = experiments_df[
    (experiments_df['LearningMode'] == 'inference') &
    (experiments_df['Accuracy'].notna())
].copy()

print(f"Found {len(inference_df)} inference experiments")
print(f"\nInference experiments breakdown:")
print(inference_df.groupby(['Dataset_Clean', 'Transfer_Clean', 'Model_Clean'])['Accuracy'].agg(['mean', 'count']))

# Create transfer matrix for each model type
datasets = ['NASA', 'NATURE2024', 'PHM2010']

def create_transfer_matrix(model_type):
    """Create transfer matrix for a specific model type."""
    matrix = np.zeros((len(datasets), len(datasets)))
    matrix[:] = np.nan  # Initialize with NaN
    
    model_df = inference_df[inference_df['Model_Clean'] == model_type]
    
    for _, row in model_df.iterrows():
        source = row['Dataset_Clean']
        target = row['Transfer_Clean']
        
        if source in datasets and target in datasets:
            i = datasets.index(source)
            j = datasets.index(target)
            matrix[i, j] = row['Accuracy']
    
    return matrix

conv_matrix = create_transfer_matrix('1D Conv')
lstm_matrix = create_transfer_matrix('LSTM')

print("\n1D Conv Transfer Matrix:")
print(pd.DataFrame(conv_matrix, index=datasets, columns=datasets))
print("\nLSTM Transfer Matrix:")
print(pd.DataFrame(lstm_matrix, index=datasets, columns=datasets))

In [ ]:
# Create Figure 2: Zero-Shot Transfer Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Function to plot heatmap with annotations
def plot_transfer_heatmap(ax, matrix, title, cbar_label=True):
    # Create mask for diagonal (no within-domain transfers)
    mask = np.eye(len(datasets), dtype=bool)
    
    # Plot heatmap
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0.2, vmax=0.9)
    
    # Set ticks and labels
    ax.set_xticks(np.arange(len(datasets)))
    ax.set_yticks(np.arange(len(datasets)))
    ax.set_xticklabels(datasets)
    ax.set_yticklabels(datasets)
    ax.set_xlabel('Target Domain', fontweight='bold')
    ax.set_ylabel('Source Domain', fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=15)
    
    # Rotate x-axis labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    # Add text annotations with accuracy values
    for i in range(len(datasets)):
        for j in range(len(datasets)):
            if not mask[i, j] and not np.isnan(matrix[i, j]):
                text_color = 'white' if matrix[i, j] < 0.5 else 'black'
                text = ax.text(j, i, f'{matrix[i, j]:.2f}',
                             ha="center", va="center", color=text_color,
                             fontsize=10, fontweight='bold')
            elif mask[i, j]:
                # Mark diagonal (within-domain) as N/A
                ax.text(j, i, 'N/A',
                       ha="center", va="center", color='gray',
                       fontsize=9, style='italic')
    
    # Add colorbar
    if cbar_label:
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Accuracy', rotation=270, labelpad=20, fontweight='bold')
    
    return im

# Plot both heatmaps
plot_transfer_heatmap(axes[0], conv_matrix, '1D Conv - Zero-Shot Transfer', cbar_label=True)
plot_transfer_heatmap(axes[1], lstm_matrix, 'LSTM - Zero-Shot Transfer', cbar_label=True)

plt.tight_layout()

# Save figure
fig2_path = os.path.join(figures_output_dir, 'figure2_zero_shot_transfer_heatmap.pdf')
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
fig2_png_path = os.path.join(figures_output_dir, 'figure2_zero_shot_transfer_heatmap.png')
plt.savefig(fig2_png_path, dpi=300, bbox_inches='tight')

print(f"✓ Figure 2 saved to: {fig2_path}")
plt.show()

## 7. Figure 3: Transfer Learning Improvement Analysis

This figure shows the relationship between zero-shot and fine-tuned performance, illustrating the efficacy of transfer learning adaptation.

In [ ]:
# Filter for transfer learning experiments
transfer_learning_df = experiments_df[
    (experiments_df['LearningMode'] == 'transferLearning') &
    (experiments_df['Accuracy'].notna())
].copy()

print(f"Found {len(transfer_learning_df)} transfer learning experiments")

# For each transfer learning experiment, find the corresponding inference (zero-shot) result
transfer_comparison = []

for _, tl_row in transfer_learning_df.iterrows():
    source = tl_row['Dataset_Clean']
    target = tl_row['Transfer_Clean']
    model = tl_row['Model_Clean']
    
    # Find matching inference experiment
    inf_match = inference_df[
        (inference_df['Dataset_Clean'] == source) &
        (inference_df['Transfer_Clean'] == target) &
        (inference_df['Model_Clean'] == model)
    ]
    
    if not inf_match.empty:
        zero_shot_acc = inf_match['Accuracy'].mean()  # Average if multiple runs
        fine_tuned_acc = tl_row['Accuracy']
        
        # Find baseline (within-domain) performance for target
        baseline_match = base_training_df[
            (base_training_df['Dataset_Clean'] == target) &
            (base_training_df['Model_Clean'] == model)
        ]
        baseline_acc = baseline_match['Accuracy'].mean() if not baseline_match.empty else np.nan
        
        transfer_comparison.append({
            'Source': source,
            'Target': target,
            'Model': model,
            'Zero_Shot': zero_shot_acc,
            'Fine_Tuned': fine_tuned_acc,
            'Baseline': baseline_acc,
            'Absolute_Gain': fine_tuned_acc - zero_shot_acc,
            'Remaining_Gap': baseline_acc - fine_tuned_acc if not np.isnan(baseline_acc) else np.nan
        })

transfer_comparison_df = pd.DataFrame(transfer_comparison)

print(f"\nCreated comparison for {len(transfer_comparison_df)} transfer scenarios")
print("\nTransfer Learning Statistics:")
print(f"Median absolute gain: {transfer_comparison_df['Absolute_Gain'].median():.3f}")
print(f"Median remaining gap: {transfer_comparison_df['Remaining_Gap'].median():.3f}")
print(f"Negative transfer cases: {(transfer_comparison_df['Absolute_Gain'] < 0).sum()}")

transfer_comparison_df

In [ ]:
# Create Figure 3: Transfer Learning Improvement Scatterplot
fig, ax = plt.subplots(figsize=(10, 8))

# Define markers and colors for different source datasets
source_markers = {'NASA': 'o', 'NATURE2024': '^', 'PHM2010': 's'}
model_colors = {'1D Conv': COLOR_1D_CONV, 'LSTM': COLOR_LSTM}

# Plot each point with appropriate marker and color
for _, row in transfer_comparison_df.iterrows():
    marker = source_markers[row['Source']]
    color = model_colors[row['Model']]
    
    # Plot the point
    ax.scatter(row['Zero_Shot'], row['Fine_Tuned'], 
              marker=marker, color=color, s=120, alpha=0.7, 
              edgecolors='black', linewidth=1.5)
    
    # Add arrow showing improvement (if significant)
    if abs(row['Absolute_Gain']) > 0.02:  # Only show arrows for changes > 2%
        arrow_color = 'green' if row['Absolute_Gain'] > 0 else 'red'
        ax.annotate('', xy=(row['Zero_Shot'], row['Fine_Tuned']),
                   xytext=(row['Zero_Shot'], row['Zero_Shot']),
                   arrowprops=dict(arrowstyle='->', color=arrow_color, 
                                 lw=1.5, alpha=0.6))

# Add diagonal reference line (no improvement)
lim_min = min(transfer_comparison_df['Zero_Shot'].min(), 
              transfer_comparison_df['Fine_Tuned'].min()) - 0.05
lim_max = max(transfer_comparison_df['Zero_Shot'].max(), 
              transfer_comparison_df['Fine_Tuned'].max()) + 0.05
ax.plot([lim_min, lim_max], [lim_min, lim_max], 
        'k--', linewidth=1.5, alpha=0.5, label='No Improvement')

# Add line for median improvement
median_gain = transfer_comparison_df['Absolute_Gain'].median()
ax.plot([lim_min, lim_max], [lim_min + median_gain, lim_max + median_gain],
        'b:', linewidth=2, alpha=0.7, label=f'Median Gain (+{median_gain:.2f})')

# Customize plot
ax.set_xlabel('Zero-Shot Accuracy', fontweight='bold', fontsize=12)
ax.set_ylabel('Fine-Tuned Accuracy', fontweight='bold', fontsize=12)
ax.set_title('Transfer Learning Efficacy: Fine-Tuned vs Zero-Shot Performance', 
            fontweight='bold', pad=15, fontsize=13)
ax.set_xlim([lim_min, lim_max])
ax.set_ylim([lim_min, lim_max])
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_aspect('equal', adjustable='box')

# Create custom legends
# Legend 1: Source datasets (markers)
source_handles = [plt.Line2D([0], [0], marker=marker, color='w', 
                            markerfacecolor='gray', markersize=10, 
                            label=source, markeredgecolor='black', markeredgewidth=1)
                 for source, marker in source_markers.items()]

# Legend 2: Model types (colors)
model_handles = [plt.Line2D([0], [0], marker='o', color='w', 
                           markerfacecolor=color, markersize=10, 
                           label=model, markeredgecolor='black', markeredgewidth=1)
                for model, color in model_colors.items()]

# Add legends
legend1 = ax.legend(handles=source_handles, title='Source Dataset', 
                   loc='upper left', framealpha=0.9, fontsize=9)
legend2 = ax.legend(handles=model_handles, title='Model Type', 
                   loc='lower right', framealpha=0.9, fontsize=9)
ax.add_artist(legend1)  # Add first legend back

# Add reference line legend
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), 
         ncol=2, framealpha=0.9, fontsize=9)
ax.add_artist(legend1)
ax.add_artist(legend2)

plt.tight_layout()

# Save figure
fig3_path = os.path.join(figures_output_dir, 'figure3_transfer_learning_improvement.pdf')
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
fig3_png_path = os.path.join(figures_output_dir, 'figure3_transfer_learning_improvement.png')
plt.savefig(fig3_png_path, dpi=300, bbox_inches='tight')

print(f"✓ Figure 3 saved to: {fig3_path}")
plt.show()

## 8. Generate Summary Tables for Paper

Create formatted tables for inclusion in the LaTeX paper.

In [ ]:
# Table 1: Zero-Shot Transfer Results (for Section 5.2)
zero_shot_table = []

for source in datasets:
    for target in datasets:
        if source != target:  # Exclude within-domain
            for model in ['1D Conv', 'LSTM']:
                matches = inference_df[
                    (inference_df['Dataset_Clean'] == source) &
                    (inference_df['Transfer_Clean'] == target) &
                    (inference_df['Model_Clean'] == model)
                ]
                
                if not matches.empty:
                    acc = matches['Accuracy'].mean()
                    std = matches['Accuracy'].std() if len(matches) > 1 else 0
                    
                    # Find baseline for gap calculation
                    baseline = base_stats[
                        (base_stats['Dataset_Clean'] == target) &
                        (base_stats['Model_Clean'] == model)
                    ]
                    gap = baseline['mean'].values[0] - acc if not baseline.empty else np.nan
                    
                    zero_shot_table.append({
                        'Source': source,
                        'Target': target,
                        'Model': model,
                        'Accuracy': f"{acc:.2f} ± {std:.2f}",
                        'Gap': f"({gap:+.2f})" if not np.isnan(gap) else "N/A"
                    })

zero_shot_table_df = pd.DataFrame(zero_shot_table)
print("Zero-Shot Transfer Results Table:")
print(zero_shot_table_df.to_string(index=False))

# Save as CSV
table1_path = os.path.join(figures_output_dir, 'table_zero_shot_results.csv')
zero_shot_table_df.to_csv(table1_path, index=False)
print(f"\n✓ Table saved to: {table1_path}")

In [ ]:
# Table 2: Transfer Learning Results (for Section 5.3)
tl_summary = transfer_comparison_df.copy()
tl_summary['Zero_Shot_Fmt'] = tl_summary['Zero_Shot'].apply(lambda x: f"{x:.2f}")
tl_summary['Fine_Tuned_Fmt'] = tl_summary['Fine_Tuned'].apply(lambda x: f"{x:.2f}")
tl_summary['Absolute_Gain_Fmt'] = tl_summary['Absolute_Gain'].apply(
    lambda x: f"{x:+.2f}" + (" ↓" if x < 0 else "")
)
tl_summary['Remaining_Gap_Fmt'] = tl_summary['Remaining_Gap'].apply(
    lambda x: f"{x:.2f}" if not np.isnan(x) else "N/A"
)

tl_table = tl_summary[[
    'Source', 'Target', 'Model', 
    'Zero_Shot_Fmt', 'Fine_Tuned_Fmt', 
    'Absolute_Gain_Fmt', 'Remaining_Gap_Fmt'
]].rename(columns={
    'Zero_Shot_Fmt': 'Zero-Shot',
    'Fine_Tuned_Fmt': 'Fine-Tuned',
    'Absolute_Gain_Fmt': 'Gain',
    'Remaining_Gap_Fmt': 'Gap to Baseline'
})

print("Transfer Learning Results Table:")
print(tl_table.to_string(index=False))

# Save as CSV
table2_path = os.path.join(figures_output_dir, 'table_transfer_learning_results.csv')
tl_table.to_csv(table2_path, index=False)
print(f"\n✓ Table saved to: {table2_path}")

## 9. Final Summary and Export

Summary of all generated figures and tables.

In [ ]:
print("="*70)
print("FIGURE GENERATION COMPLETE")
print("="*70)
print(f"\nAll figures and tables saved to: {figures_output_dir}")
print("\nGenerated Files:")
print("  📊 Figure 1: figure1_base_training_performance.pdf/png")
print("  📊 Figure 2: figure2_zero_shot_transfer_heatmap.pdf/png")
print("  📊 Figure 3: figure3_transfer_learning_improvement.pdf/png")
print("  📋 Table 1: table_zero_shot_results.csv")
print("  📋 Table 2: table_transfer_learning_results.csv")

print("\nKey Findings Summary:")
print(f"  • Base Training: PHM2010 easiest (CNN: {base_stats[base_stats['Dataset_Clean']=='PHM2010']['mean'].values[0]:.2f})")
print(f"  • Zero-Shot: {(inference_df['Accuracy'] < 0.5).sum()}/{len(inference_df)} transfers below random")
print(f"  • Fine-Tuning: Median gain {transfer_comparison_df['Absolute_Gain'].median():.2f}, "
      f"{(transfer_comparison_df['Absolute_Gain'] < 0).sum()}/{len(transfer_comparison_df)} negative transfer")
print("\n" + "="*70)
print("✓ Ready for paper integration")
print("="*70)

## 10. Figure 4: Regression Baseline Performance

R² score for all six dataset × architecture combinations under continuous wear prediction (regression formulation). Negative R² indicates the model performs worse than predicting the mean, confirming that target variable semantic structure — not model capacity — drives failure on NATURE2024 and PHM2010. Supports Section 5.5 (Target Variable Gaps).

In [ ]:

import glob

# Paths to new experiment outputs and Paper figures directory
code_base = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption'
paper_figures_dir = '/Users/flore/Documents/01_Forschung/05_Publikationen/04_digitalTwin/Paper/figures'
os.makedirs(paper_figures_dir, exist_ok=True)

# ── Load regression results ──────────────────────────────────────────────────
reg_experiments = {
    ('NASA',       '1D Conv'): os.path.join(code_base, 'experiments_regression', 'nasa_1dconv_regression'),
    ('NASA',       'LSTM'):    os.path.join(code_base, 'experiments_regression', 'nasa_lstm_regression'),
    ('NATURE2024', '1D Conv'): os.path.join(code_base, 'experiments_regression', 'nature_1dconv_regression'),
    ('NATURE2024', 'LSTM'):    os.path.join(code_base, 'experiments_regression', 'nature_lstm_regression'),
    ('PHM2010',    '1D Conv'): os.path.join(code_base, 'experiments_regression', 'phm_1dconv_regression'),
    ('PHM2010',    'LSTM'):    os.path.join(code_base, 'experiments_regression', 'phm_lstm_regression'),
}

reg_results = {}
for (dataset, model), exp_dir in reg_experiments.items():
    pattern = os.path.join(exp_dir, 'logs', 'evaluation_regression_train_*.csv')
    files = sorted(glob.glob(pattern))
    if files:
        df = pd.read_csv(files[-1])
        reg_results[(dataset, model)] = {
            'MAE':  df['MAE'].iloc[-1],
            'RMSE': df['RMSE'].iloc[-1],
            'R2':   df['R2'].iloc[-1],
        }
        r2 = reg_results[(dataset, model)]['R2']
        print(f"  {dataset:12s} {model:8s}: R²={r2:+.3f}")
    else:
        print(f"  ⚠ No file found: {exp_dir}")

print(f"\nLoaded {len(reg_results)}/6 regression results")

# ── Figure 4: Regression Baseline Performance ────────────────────────────────
datasets_reg = ['NASA', 'NATURE2024', 'PHM2010']
models_reg = ['1D Conv', 'LSTM']
x = np.arange(len(datasets_reg))
bar_width = 0.32

fig, ax = plt.subplots(figsize=(9, 5))

r2_conv = [reg_results.get((d, '1D Conv'), {}).get('R2', np.nan) for d in datasets_reg]
r2_lstm = [reg_results.get((d, 'LSTM'),    {}).get('R2', np.nan) for d in datasets_reg]

bars1 = ax.bar(x - bar_width/2, r2_conv, bar_width,
               label='1D Conv', color=COLOR_1D_CONV, alpha=0.85)
bars2 = ax.bar(x + bar_width/2, r2_lstm, bar_width,
               label='LSTM', color=COLOR_LSTM, alpha=0.85)

# Reference line at R²=0
ax.axhline(y=0, color='black', linestyle='-', linewidth=1.0, alpha=0.8)
# Reference line at R²=1 (perfect)
ax.axhline(y=1, color=COLOR_BASELINE, linestyle=':', linewidth=1.0, alpha=0.5, label='Perfect (R²=1)')

# Value labels (clipped to axis to avoid very negative values going off-chart)
def label_r2_bars(bars, values):
    for bar, val in zip(bars, values):
        if np.isnan(val):
            continue
        y_label = max(val, ax.get_ylim()[0]) if val < 0 else val
        va = 'top' if val < 0 else 'bottom'
        offset = -0.05 if val < 0 else 0.05
        ax.text(bar.get_x() + bar.get_width() / 2,
                max(val, -3.5) + offset,
                f'{val:+.2f}',
                ha='center', va=va, fontsize=8, fontweight='bold',
                color='white' if val < -1 else 'black')

ax.set_ylim([-4, 1.2])  # clip display (PHM CNN -23 shown as clipped bar)
label_r2_bars(bars1, r2_conv)
label_r2_bars(bars2, r2_lstm)

# Add note for heavily clipped values
for bar, val in zip(list(bars1) + list(bars2), r2_conv + r2_lstm):
    if val < -4:
        ax.text(bar.get_x() + bar.get_width() / 2, -3.7,
                f'({val:.1f})', ha='center', va='top', fontsize=7,
                color='dimgray', style='italic')

ax.set_xlabel('Dataset', fontweight='bold')
ax.set_ylabel('R² Score', fontweight='bold')
ax.set_title('Regression Baseline Performance by Dataset and Architecture',
             fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(datasets_reg)
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle=':')
plt.tight_layout()

# Save
fig4_pdf = os.path.join(paper_figures_dir, 'figure4_regression_baseline.pdf')
fig4_png = os.path.join(figures_output_dir, 'figure4_regression_baseline.png')
plt.savefig(fig4_pdf, dpi=300, bbox_inches='tight')
plt.savefig(fig4_png, dpi=300, bbox_inches='tight')
print(f"\n✓ Figure 4 saved to: {fig4_pdf}")
plt.show()


## 11. Figure 6: Multi-Channel vs. Single-Channel Accuracy

Compares classification accuracy when using only the RMS-reduced single channel (standard pipeline) versus full multi-axis vibration channels, supporting the Sensor Dimensionality Gap analysis (Section 5.6).

In [ ]:

import glob

# Paths
code_base = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption'
paper_figures_dir = '/Users/flore/Documents/01_Forschung/05_Publikationen/04_digitalTwin/Paper/figures'
os.makedirs(paper_figures_dir, exist_ok=True)

# ── Load multi-channel classification results ───────────────────────────────
mc_experiments = {
    ('NATURE2024', '1D Conv', '2ch'): os.path.join(code_base, 'experiments_multichannel', 'nature_2ch_1dconv'),
    ('NATURE2024', 'LSTM',    '2ch'): os.path.join(code_base, 'experiments_multichannel', 'nature_2ch_lstm'),
    ('PHM2010',   '1D Conv', '3ch'): os.path.join(code_base, 'experiments_multichannel', 'phm_3ch_1dconv'),
    ('PHM2010',   'LSTM',    '3ch'): os.path.join(code_base, 'experiments_multichannel', 'phm_3ch_lstm'),
}

mc_results = {}
for (dataset, model, channels), exp_dir in mc_experiments.items():
    pattern = os.path.join(exp_dir, 'logs', 'evaluation_results_train_*.csv')
    files = sorted(glob.glob(pattern))
    if files:
        df = pd.read_csv(files[-1])
        mc_results[(dataset, model, channels)] = {
            'Accuracy': df['Accuracy'].iloc[-1],
            'F1': df['F1-Score'].iloc[-1],
        }
        print(f"  {dataset} {model} {channels}: Acc={mc_results[(dataset, model, channels)]['Accuracy']:.3f}")
    else:
        print(f"  ⚠ No file found: {exp_dir}")

# ── Single-channel baselines from summary CSV ────────────────────────────────
# These are the standard within-dataset baselines (same train/test split)
single_ch_baselines = {
    ('NATURE2024', '1D Conv'): base_stats.loc[
        (base_stats['Dataset_Clean']=='NATURE2024') & (base_stats['Model_Clean']=='1D Conv'), 'mean'].values[0],
    ('NATURE2024', 'LSTM'): base_stats.loc[
        (base_stats['Dataset_Clean']=='NATURE2024') & (base_stats['Model_Clean']=='LSTM'), 'mean'].values[0],
    ('PHM2010', '1D Conv'): base_stats.loc[
        (base_stats['Dataset_Clean']=='PHM2010') & (base_stats['Model_Clean']=='1D Conv'), 'mean'].values[0],
    ('PHM2010', 'LSTM'): base_stats.loc[
        (base_stats['Dataset_Clean']=='PHM2010') & (base_stats['Model_Clean']=='LSTM'), 'mean'].values[0],
}
print("\nSingle-channel baselines:")
for k, v in single_ch_baselines.items():
    print(f"  {k[0]} {k[1]}: {v:.3f}")

# ── Figure 6: Multi-Channel vs Single-Channel ────────────────────────────────
datasets_mc = ['NATURE2024', 'PHM2010']
channel_labels = {'NATURE2024': '2-ch', 'PHM2010': '3-ch'}
models_mc = ['1D Conv', 'LSTM']
model_colors_mc = {'1D Conv': COLOR_1D_CONV, 'LSTM': COLOR_LSTM}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax_idx, dataset in enumerate(datasets_mc):
    ax = axes[ax_idx]
    n_models = len(models_mc)
    group_width = 0.7
    bar_width = group_width / (2 * n_models)
    
    for m_idx, model in enumerate(models_mc):
        # Single-channel bar (solid)
        x_single = m_idx * (group_width / n_models + 0.05)
        single_acc = single_ch_baselines[(dataset, model)]
        bar_s = ax.bar(x_single, single_acc, bar_width,
                       color=model_colors_mc[model], alpha=0.85,
                       label=f'{model} (1-ch)' if ax_idx == 0 else '_nolegend_')
        ax.text(x_single, single_acc + 0.01, f'{single_acc:.2f}',
                ha='center', va='bottom', fontsize=8)
        
        # Multi-channel bar (hatched)
        x_multi = x_single + bar_width + 0.01
        ch_key = ('NATURE2024' if dataset == 'NATURE2024' else 'PHM2010', model, '2ch' if dataset == 'NATURE2024' else '3ch')
        if ch_key in mc_results:
            multi_acc = mc_results[ch_key]['Accuracy']
            bar_m = ax.bar(x_multi, multi_acc, bar_width,
                           color=model_colors_mc[model], alpha=0.5,
                           hatch='//', edgecolor=model_colors_mc[model],
                           label=f'{model} ({channel_labels[dataset]})' if ax_idx == 0 else '_nolegend_')
            ax.text(x_multi, multi_acc + 0.01, f'{multi_acc:.2f}',
                    ha='center', va='bottom', fontsize=8)

    ax.axhline(y=0.5, color=COLOR_BASELINE, linestyle='--', linewidth=1,
               alpha=0.6, label='Random Baseline' if ax_idx == 0 else '_nolegend_')
    
    n_ticks = len(models_mc)
    tick_centers = [m_idx * (group_width / n_models + 0.05) + bar_width / 2
                    for m_idx in range(n_ticks)]
    ax.set_xticks(tick_centers)
    ax.set_xticklabels(models_mc)
    ax.set_title(f'{dataset}\n(1-ch vs {channel_labels[dataset]})', fontweight='bold')
    ax.set_ylabel('Classification Accuracy' if ax_idx == 0 else '')
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3, linestyle=':')

axes[0].legend(loc='upper left', framealpha=0.9, fontsize=8,
               title='Architecture (channels)')

fig.suptitle('Multi-Channel vs. Single-Channel Classification Accuracy',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()

# Save
fig6_pdf = os.path.join(paper_figures_dir, 'figure6_multichannel_comparison.pdf')
fig6_png = os.path.join(figures_output_dir, 'figure6_multichannel_comparison.png')
plt.savefig(fig6_pdf, dpi=300, bbox_inches='tight')
plt.savefig(fig6_png, dpi=300, bbox_inches='tight')
print(f"\n✓ Figure 6 saved to: {fig6_pdf}")
plt.show()
